# AWP Wind Lidar Planning

**Purpose:** This notebook demonstrates how to plan flights for NASA Langley's Aerosol Wind Profiler (AWP) using HyPlan. AWP is not a swath-mapping instrument like an imaging spectrometer or SAR; instead, it alternates between two off-nadir lines of sight (LOS) to derive horizontal wind-vector profiles along the flight path. That means AWP planning is driven by stable straight-and-level legs, LOS geometry, and vector-profile spacing.

The geometry and operating assumptions used here follow Bedka et al. (2024) and Bedka (2025), which are cited in the references section at the end of the notebook.

| | |
|---|---|
| **Audience** | Intermediate |
| **Runtime** | < 2 minutes |
| **Requires internet** | Optional (terrain-aware section downloads DEM) |
| **Credentials required** | None |
| **Optional dependencies** | None |
| **Uses example data** | No |

**What You Will Learn:**
- How the AWP instrument model represents dual-LOS wind-lidar geometry
- How vector-profile spacing depends on dwell time and aircraft speed
- How to predict where AWP profiles fall along a flight line
- How terrain changes AGL and LOS intercept placement along a mountainous leg
- How to flag which parts of a full flight plan are stable enough for AWP retrievals


In [ ]:
import datetime as dt

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from hyplan import (
    AerosolWindProfiler,
    FlightLine,
    NASA_GIII,
    awp_profile_locations_for_flight_line,
    awp_profile_locations_for_plan,
    compute_flight_plan,
    flag_awp_stable_segments,
    racetrack,
    ureg,
)

from hyplan.terrain import generate_demfile


## 1. AWP Instrument Overview

The HyPlan `AerosolWindProfiler` class is a planning-oriented model derived from Bedka et al. (2024) and the Bedka (2025) NOAA 3-D winds final report. It captures the geometry and timing assumptions that matter for flight design:

- **Dual LOS:** two viewing directions at `±45°` azimuth from the aircraft nose
- **Off-nadir angle:** `30°`
- **Common airborne mode:** `3 s` on LOS1, `1 s` nadir, `3 s` on LOS2
- **Nominal speed:** `225 m/s`, giving vector profiles every `~1.6 km`
- **Planning constraint:** useful wind-vector retrievals require stable flight; turns and strong altitude changes create gaps

This means AWP belongs in a different planning category than conventional swath sensors. The key questions are not "what width do I map?" but "where do my wind profiles land, and which legs are actually usable?"


In [ ]:
awp = AerosolWindProfiler()

summary = pd.DataFrame(
    [
        ("Wavelength", f"{awp.wavelength:.2f}"),
        ("Off-nadir angle", f"{awp.off_nadir_angle:.1f} deg"),
        ("Relative LOS azimuths", f"{awp.los_azimuths_relative[0]:.0f}°, {awp.los_azimuths_relative[1]:.0f}°"),
        ("Pulse rate", f"{awp.pulse_rate:.0f}"),
        ("Max slant range", f"{awp.max_slant_range().to(ureg.kilometer):.1f}"),
        ("Max vertical range", f"{awp.max_vertical_range().to(ureg.kilometer):.1f}"),
        ("Blind zone", f"{awp.blind_zone:.0f}"),
        ("Vector profile spacing @ 225 m/s", f"{awp.vector_profile_spacing(225 * ureg.meter / ureg.second).to(ureg.kilometer):.2f}"),
        ("LOS separation @ 12 km AGL", f"{awp.los_surface_separation(12 * ureg.kilometer).to(ureg.kilometer):.2f}"),
        ("Vertical bin spacing (256 FFT samples)", f"{awp.vertical_bin_spacing(256).to(ureg.meter):.1f}"),
        ("Vertical bin spacing (512 FFT samples)", f"{awp.vertical_bin_spacing(512).to(ureg.meter):.1f}"),
    ],
    columns=["Parameter", "Value"],
)

summary


## 2. Profile Locations Along a Single Flight Line

For a single straight survey leg, HyPlan can place the expected AWP vector profiles along the flight path and compute the two LOS ground intercepts associated with each sample location.

Below we make a simple northbound `80 km` line at `12 km` altitude and ask where AWP would place vector profiles if the aircraft were cruising at `225 m/s`.


In [ ]:
flight_line = FlightLine.start_length_azimuth(
    lat1=34.0,
    lon1=-118.2,
    length=80 * ureg.kilometer,
    az=0.0,
    altitude_msl=12 * ureg.kilometer,
    site_name="AWP Example Leg",
)

profiles = awp_profile_locations_for_flight_line(
    flight_line,
    ground_speed=225 * ureg.meter / ureg.second,
    start_time=dt.datetime(2025, 10, 15, 17, 0, 0),
    altitude_agl=12 * ureg.kilometer,
)

profiles[[
    "sample_index",
    "distance_from_start_m",
    "elapsed_time_s",
    "time_utc",
    "platform_heading_deg",
    "los_surface_separation_m",
]].head(8)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 8))

ax.plot([flight_line.lon1, flight_line.lon2], [flight_line.lat1, flight_line.lat2], color="black", linewidth=2, label="Flight line")
ax.scatter(profiles["platform_lon"], profiles["platform_lat"], s=16, color="#2563eb", label="Vector profile")
ax.scatter(profiles["los1_lon"], profiles["los1_lat"], s=8, alpha=0.45, color="#16a34a", label="LOS1 intercept")
ax.scatter(profiles["los2_lon"], profiles["los2_lat"], s=8, alpha=0.45, color="#dc2626", label="LOS2 intercept")

ax.set_title("AWP profile locations and LOS ground intercepts")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend(loc="best")
ax.grid(alpha=0.25)
plt.show()


## 3. Terrain-Aware LOS Placement

A fixed flight altitude MSL does **not** imply a fixed AGL over mountains. In `terrain_aware` mode, HyPlan samples terrain beneath each planned profile location, computes the local AGL, and ray-traces both AWP beams into a DEM to estimate terrain-adjusted LOS intercepts.

This example uses a line across the San Gabriel Mountains. The DEM is downloaded on first run and then reused from the local cache.


In [ ]:
mountain_line = FlightLine.start_length_azimuth(
    lat1=34.15,
    lon1=-118.05,
    length=55 * ureg.kilometer,
    az=45.0,
    altitude_msl=6.5 * ureg.kilometer,
    site_name="San Gabriel Terrain Leg",
)

terrain_dem = generate_demfile(
    np.array([mountain_line.lat1, mountain_line.lat2]),
    np.array([mountain_line.lon1, mountain_line.lon2]),
)

terrain_profiles = awp_profile_locations_for_flight_line(
    mountain_line,
    ground_speed=225 * ureg.meter / ureg.second,
    start_time=dt.datetime(2025, 10, 15, 18, 0, 0),
    terrain_aware=True,
    dem_file=terrain_dem,
)

terrain_profiles[[
    "sample_index",
    "terrain_elevation_m",
    "altitude_agl_m",
    "los_surface_separation_m",
    "los1_alt_m",
    "los2_alt_m",
]].head(8)


In [ ]:
distance_km = terrain_profiles["distance_from_start_m"] / 1000.0
flight_altitude_m = mountain_line.altitude_msl.to("meter").magnitude

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 8), sharex=True)

ax1.plot(distance_km, terrain_profiles["terrain_elevation_m"], color="saddlebrown", linewidth=2, label="Terrain beneath aircraft")
ax1.axhline(flight_altitude_m, color="black", linestyle="--", linewidth=1.5, label="Aircraft altitude MSL")
ax1.set_ylabel("Elevation (m)")
ax1.set_title("Terrain-aware AWP profile geometry")
ax1.grid(alpha=0.25)
ax1.legend(loc="best")

ax2.plot(distance_km, terrain_profiles["altitude_agl_m"], color="#2563eb", linewidth=2, label="Aircraft altitude AGL")
ax2.set_xlabel("Distance along flight line (km)")
ax2.set_ylabel("Altitude AGL (m)", color="#2563eb")
ax2.tick_params(axis="y", labelcolor="#2563eb")
ax2.grid(alpha=0.25)

ax2b = ax2.twinx()
ax2b.plot(distance_km, terrain_profiles["los_surface_separation_m"] / 1000.0, color="#16a34a", linewidth=2, label="LOS surface separation")
ax2b.set_ylabel("LOS separation (km)", color="#16a34a")
ax2b.tick_params(axis="y", labelcolor="#16a34a")

lines = ax2.get_lines() + ax2b.get_lines()
labels = [line.get_label() for line in lines]
ax2.legend(lines, labels, loc="best")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

ax.plot(
    [mountain_line.lon1, mountain_line.lon2],
    [mountain_line.lat1, mountain_line.lat2],
    color="black",
    linewidth=2,
    label="Flight line",
)

sc = ax.scatter(
    terrain_profiles["platform_lon"],
    terrain_profiles["platform_lat"],
    c=terrain_profiles["altitude_agl_m"],
    cmap="viridis",
    s=28,
    edgecolor="white",
    linewidth=0.4,
    label="Terrain-aware profile location",
)

ax.scatter(
    terrain_profiles["los1_lon"],
    terrain_profiles["los1_lat"],
    s=10,
    alpha=0.5,
    color="#16a34a",
    label="LOS1 terrain intercept",
)
ax.scatter(
    terrain_profiles["los2_lon"],
    terrain_profiles["los2_lat"],
    s=10,
    alpha=0.5,
    color="#dc2626",
    label="LOS2 terrain intercept",
)

cbar = plt.colorbar(sc, ax=ax, pad=0.02)
cbar.set_label("Aircraft altitude AGL (m)")

ax.set_title("Terrain-aware AWP profile locations and LOS intercepts")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.grid(alpha=0.25)
ax.legend(loc="best")
plt.show()


**What changed here?** Terrain raises and lowers the effective AGL even though the aircraft stays at a fixed `6.5 km` MSL. Because the terrain-aware mode ray-traces each LOS into the DEM, the surface intercept separation also varies along the leg instead of staying fixed at a flat-earth value. That makes this mode a better planning approximation for mountainous wind-profiling sorties.


## 4. Dwell-Time Sensitivity

AWP planning has a strong tradeoff between **horizontal resolution** and **signal integration**. Longer dwell per LOS improves sensitivity, but pushes vector profiles farther apart.

This is a natural planning parameter to explore in HyPlan because it directly changes the spacing of expected profile locations.


In [ ]:
rows = []
for dwell_s in [2, 3, 6, 15]:
    dwell = dwell_s * ureg.second
    rows.append(
        {
            "Dwell per LOS (s)": dwell_s,
            "Pulses per LOS": int((awp.pulse_rate * dwell).to_base_units().magnitude),
            "Time between vector profiles (s)": awp.vector_profile_time_spacing(
                dwell_time_per_los=dwell,
                nadir_dwell_time=1 * ureg.second,
            ).m_as("second"),
            "Distance between vector profiles (km)": awp.vector_profile_spacing(
                225 * ureg.meter / ureg.second,
                dwell_time_per_los=dwell,
                nadir_dwell_time=1 * ureg.second,
            ).m_as("kilometer"),
            "Distance to first profile (km)": awp.profile_assignment_offset(
                225 * ureg.meter / ureg.second,
                dwell_time_per_los=dwell,
                nadir_dwell_time=1 * ureg.second,
            ).m_as("kilometer"),
        }
    )

pd.DataFrame(rows)


## 5. AWP Along a Multi-Leg Flight Plan

The more useful planning mode is to ask: **Which segments of a full sortie are stable enough for AWP to retrieve vector winds, and where would the profiles appear?**

The AWP helper functions treat turns, climbs, descents, and segments with strong heading or altitude changes as unstable. This mirrors the operational reality described in the report: the best AWP data comes from long, straight, stable legs.


In [ ]:
pattern = racetrack(
    center=(34.1, -118.2),
    heading=90.0,
    altitude=12 * ureg.kilometer,
    leg_length=70 * ureg.kilometer,
    n_legs=4,
    offset=12 * ureg.kilometer,
)

plan = compute_flight_plan(
    aircraft=NASA_GIII(),
    flight_sequence=[pattern],
)

flagged = flag_awp_stable_segments(plan)
flagged[[
    "segment_type",
    "segment_name",
    "distance",
    "time_to_segment",
    "awp_heading_change_deg",
    "awp_altitude_change_m",
    "awp_stable_platform_ok",
]].head(12)


In [ ]:
profiles_plan = awp_profile_locations_for_plan(
    plan,
    takeoff_time=dt.datetime(2025, 10, 15, 17, 0, 0),
)

profiles_plan.groupby(["source_segment_type", "source_segment_name"]).size().rename("n_profiles")


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

for _, row in flagged.iterrows():
    geom = row.geometry
    if getattr(geom, "geom_type", None) != "LineString":
        continue
    xs, ys = geom.xy
    color = "#16a34a" if row["awp_stable_platform_ok"] else "#9ca3af"
    width = 2.5 if row["segment_type"] == "flight_line" else 1.5
    ax.plot(xs, ys, color=color, linewidth=width, alpha=0.9)

ax.scatter(
    profiles_plan.geometry.x,
    profiles_plan.geometry.y,
    s=14,
    color="#2563eb",
    label="AWP vector profiles",
)

ax.set_title("Stable plan segments and predicted AWP profile locations")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.grid(alpha=0.25)
ax.legend(loc="best")
plt.show()


## Operational Takeaways

- **AWP is profile-driven, not swath-driven.** The primary design outputs are profile spacing, LOS intercept separation, and stable-leg availability.
- **Longer LOS dwell increases sensitivity but reduces horizontal detail.** HyPlan makes that tradeoff explicit in distance/time units.
- **Terrain-aware AWP planning changes both AGL and beam intercept geometry.** Over relief, DEM-backed ray tracing gives more realistic surface intercepts than a flat-earth AGL assumption.
- **Turns are the enemy of clean vector retrievals.** AWP planning works best when you can create long straight legs with minimal roll and altitude change.
- **Racetrack-style patterns are natural for repeated wind profiling.** They provide multiple stable legs while keeping the sortie geographically compact.

This notebook focuses on airborne AWP geometry. It does **not** attempt to model coherent Doppler retrieval physics, aerosol signal availability, or retrieval success probability, which should remain separate higher-fidelity analyses.


## References

Bedka, K., Marketon, J., Henderson, S., and Kavaya, M. (2024). "AWP: NASA's Aerosol Wind Profiler Coherent Doppler Wind Lidar." In Singh, U. N., Tzeremes, G., Refaat, T. F., and Ribes Pleguezuelo, P. (eds), *Space-based Lidar Remote Sensing Techniques and Emerging Technologies*, LIDAR 2023, Springer Aerospace Technology. https://doi.org/10.1007/978-3-031-53618-2_3

Bedka, K. (2025). *3-D Lidar Wind Airborne Profiling Using A Coherent-Detection Doppler Wind Lidar Designed For Space-Based Operation*. Final Project Report in response to the NOAA Broad Agency Announcement, "Measuring the Atmospheric Wind Profile (3D Winds): Call for Studies and Field Measurement Campaigns."
